In [2]:
import pandas as pd

import os

from tensorflow.keras.layers import TextVectorization
from tensorflow.keras.callbacks import EarlyStopping



processed_train = pd.read_csv('../data/processed/processed_train.csv')
processed_test = pd.read_csv('../data/processed/processed_test.csv')

# processed_train.fillna("",inplace=True)
# processed_test.fillna("",inplace=True)

x_train = processed_train['clean_comment'].astype(str).to_numpy()
x_test = processed_test['clean_comment'].astype(str).to_numpy()

y_train = processed_train['category'].map({-1:0,0:1,1:2}).astype("int32").to_numpy()
y_test = processed_test['category'].map({-1:0,0:1,1:2}).astype("int32").to_numpy()



vectorizer = TextVectorization(
    max_tokens=1000,
    output_mode='int',
    output_sequence_length=250
)

vectorizer.adapt(x_train)



from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, Dropout, Input, Bidirectional

from tensorflow.keras.layers import LSTM, GRU

early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

model = Sequential([
    Input(shape=(1,),dtype='string'),
    vectorizer,
    Embedding(10000, 64),
    Bidirectional(GRU(28)),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(28,activation='tanh'),
    Dropout(0.2),
    Dense(3, activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.fit(x_train, y_train, epochs=10, batch_size=32, validation_data=(x_test, y_test), callbacks=[early_stop])

os.makedirs('artifacts/models', exist_ok=True)
model.save('artifacts/models/comment_classifier.keras')

Epoch 1/10
929/929 ━━━━━━━━━━━━━━━━━━━━ 137s 142ms/step - accuracy: 0.7304 - loss: 0.6681 - val_accuracy: 0.8218 - val_loss: 0.5069
Epoch 2/10
929/929 ━━━━━━━━━━━━━━━━━━━━ 108s 116ms/step - accuracy: 0.8297 - loss: 0.4947 - val_accuracy: 0.8223 - val_loss: 0.5007
Epoch 3/10
929/929 ━━━━━━━━━━━━━━━━━━━━ 136s 147ms/step - accuracy: 0.8335 - loss: 0.4733 - val_accuracy: 0.8217 - val_loss: 0.4953
Epoch 4/10
152/929 ━━━━━━━━━━━━━━━━━━━━ 1:33 120ms/step - accuracy: 0.8374 - loss: 0.4648

KeyboardInterrupt: 

In [3]:
import pandas as pd

processed_train = pd.read_csv('../data/processed/processed_train.csv')
processed_test = pd.read_csv('../data/processed/processed_test.csv')

x_train = processed_train['clean_comment'].astype(str).to_numpy()
x_test = processed_test['clean_comment'].astype(str).to_numpy() 

y_train = processed_train['category'].map({-1:0,0:1,1:2}).astype("int32").to_numpy()
y_test = processed_test['category'].map({-1:0,0:1,1:2}).astype("int32").to_numpy()

from tensorflow.keras.layers import TextVectorization



vectorizer = TextVectorization(
    max_tokens=10000,
    output_mode='int',
    output_sequence_length=250
)

vectorizer.adapt(x_train)



from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, Dropout, Input, Bidirectional

from tensorflow.keras.layers import LSTM, GRU

model = Sequential([
    Input(shape=(1,),dtype='string'),
    vectorizer,
    Embedding(10000, 64),
    Bidirectional(GRU(28)),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(28,activation='tanh'),
    Dropout(0.2),
    Dense(3, activation='softmax')
])


model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model.fit(x_train, y_train, epochs=5, batch_size=32, validation_data=(x_test, y_test))



Epoch 1/5
929/929 ━━━━━━━━━━━━━━━━━━━━ 107s 109ms/step - accuracy: 0.7526 - loss: 0.5966 - val_accuracy: 0.8688 - val_loss: 0.3652
Epoch 2/5
929/929 ━━━━━━━━━━━━━━━━━━━━ 116s 124ms/step - accuracy: 0.9049 - loss: 0.2808 - val_accuracy: 0.8954 - val_loss: 0.3062
Epoch 3/5
929/929 ━━━━━━━━━━━━━━━━━━━━ 118s 127ms/step - accuracy: 0.9310 - loss: 0.2150 - val_accuracy: 0.8952 - val_loss: 0.3197
Epoch 4/5
929/929 ━━━━━━━━━━━━━━━━━━━━ 169s 182ms/step - accuracy: 0.9442 - loss: 0.1730 - val_accuracy: 0.8843 - val_loss: 0.3786
Epoch 5/5
929/929 ━━━━━━━━━━━━━━━━━━━━ 228s 245ms/step - accuracy: 0.9597 - loss: 0.1312 - val_accuracy: 0.8650 - val_loss: 0.4445


In [7]:
import mlflow
from tensorflow import keras

In [8]:
mlflow.set_tracking_uri("http://52.66.145.172:5000")

In [10]:
model = mlflow.keras.load_model(model_uri = "models:/yt-comment-analyzer/10")

In [19]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization              │ (None, 400)            │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 400, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        74,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,138,379 (15.79 MB)

 Trainable params: 1,379,459 (5.26 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2,758,920 (10.52 MB)

In [21]:
import pandas as pd
processed_test = pd.read_csv('../data/processed/processed_test.csv')    
x_test = processed_test['clean_comment'].astype(str).to_numpy()

In [28]:
list = ['This is a great video! I learned so much.',"Very awful content, waste of time.","Not bad, but could be better."]

In [30]:
example = pd.Series(list).astype(str).to_numpy()

In [33]:
example

array(['This is a great video! I learned so much.',
       'Very awful content, waste of time.',
       'Not bad, but could be better.'], dtype=object)

In [38]:
import numpy as np
preds = model.predict(example)
predictions = np.argmax(preds, axis=1)  

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step


In [41]:
predictions

array([2, 0, 0])